# HKM Top10 Threshold Classifier

This notebook reuses the top-30 HKM search outputs from `05-3_hkm_top_atoms_ranked.ipynb`.
It does not launch any enumeration. It only reads the saved HKM result files after `05-3` has finished.

Classifier rule:
- For each `(length, logic)` pair, take the top-10 HKMs.
- For one patient, compute the matched HKM ratio: `matched_top_hkms / top_k_used`.
- If the ratio is greater than or equal to the threshold, predict `label0`; otherwise predict `label1`.
- Sweep thresholds from `0.0` to `1.0` with step `0.1`.

The final figure contains 6 subplots: `3 lengths x 2 ranking logics`.
The plotted curves are test-set precision and recall for both labels.
- precision: blue
- recall: orange
- label0: solid line
- label1: dashed line


In [1]:
from pathlib import Path
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from hkm_top_atoms_ranked import (
    LOGIC_NAMES,
    TEST_IPS,
    TRAIN_IPS,
    build_output_dir,
    count_label_patients,
    load_samples,
)


In [2]:
# Input settings
TOP_ATOMS_PER_FEATURE = 30
TOP_HKMS_PER_LENGTH = 5000
TOP_K_FOR_CLASSIFIER = 10
LENGTHS = (4, 3, 2)
THRESHOLDS = np.round(np.arange(0.0, 1.01, 0.1), 2)
PLOT_SPLIT = 'test'

# HKM output directory from 05-3
OUTPUT_ROOT = Path(r'D:\PythonProject\paper\hkm_top_atoms_ranked_outputs')
OUTPUT_DIR = build_output_dir(
    top_atoms_per_feature=TOP_ATOMS_PER_FEATURE,
    top_hkms_per_length=TOP_HKMS_PER_LENGTH,
    output_root=OUTPUT_ROOT,
)

# Local output directory for this notebook
RESULT_DIR = OUTPUT_DIR / 'top10_threshold_classifier'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

LOGIC_TITLES = {
    'precision0': 'precision0',
    'precision0_plus_recall0': 'precision0_recall0_mean',
}

PLOT_CONFIG = {
    'figsize': (14, 14),
    'dpi': 160,
    'precision_color': 'tab:blue',
    'recall_color': 'tab:orange',
    'macro_f1_color': 'tab:red',
    'label0_linestyle': '-',
    'label1_linestyle': '--',
    'xlabel': 'Threshold',
    'ylabel': 'Score',
    'ylim': (0.0, 1.0),
    'grid': True,
    'grid_alpha': 0.3,
}

OUTPUT_DIR


WindowsPath('D:/PythonProject/paper/hkm_top_atoms_ranked_outputs/atoms_per_feature_30_top_hkms_5000')

In [3]:
def hkm_jsonl_path(output_dir: Path, logic_name: str, length: int, top_hkms_per_length: int) -> Path:
    return output_dir / f'hkms_{logic_name}_len{length}_top{top_hkms_per_length}.jsonl'


def load_top_hkms(output_dir: Path, logic_name: str, length: int, top_hkms_per_length: int, top_k: int):
    path = hkm_jsonl_path(output_dir, logic_name, length, top_hkms_per_length)
    if not path.exists():
        raise FileNotFoundError(
            f'Missing HKM result file: {path}. Finish 05-3 first, then rerun this notebook.'
        )
    records = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            records.append(json.loads(line))
            if len(records) >= top_k:
                break
    if not records:
        raise ValueError(f'No HKM records found in {path}')
    return records


def load_split_totals():
    train_samples = load_samples(TRAIN_IPS)
    test_samples = load_samples(TEST_IPS)
    return {
        'train': count_label_patients(train_samples),
        'test': count_label_patients(test_samples),
    }


def bit_counts_to_match_counts(records, split: str, n0: int, n1: int):
    counts0 = np.zeros(n0, dtype=np.int16)
    counts1 = np.zeros(n1, dtype=np.int16)
    key0 = f'{split}_matched_bits0_hex'
    key1 = f'{split}_matched_bits1_hex'

    for record in records:
        bits0 = int(record[key0], 16)
        bits1 = int(record[key1], 16)
        for idx in range(n0):
            counts0[idx] += (bits0 >> idx) & 1
        for idx in range(n1):
            counts1[idx] += (bits1 >> idx) & 1
    return counts0, counts1


def threshold_to_min_matches(threshold: float, top_k_used: int) -> int:
    return int(math.ceil((threshold * top_k_used) - 1e-12))


def safe_f1(precision: float, recall: float) -> float:
    return (2.0 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0


def compute_binary_metrics(tp0: int, fp0: int, n0: int, n1: int):
    fn0 = n0 - tp0
    tn0 = n1 - fp0

    precision0 = tp0 / (tp0 + fp0) if (tp0 + fp0) > 0 else 0.0
    recall0 = tp0 / n0 if n0 > 0 else 0.0

    predicted1_total = tn0 + fn0
    precision1 = tn0 / predicted1_total if predicted1_total > 0 else 0.0
    recall1 = tn0 / n1 if n1 > 0 else 0.0

    f1_0 = safe_f1(precision0, recall0)
    f1_1 = safe_f1(precision1, recall1)
    macro_f1 = (f1_0 + f1_1) / 2.0

    return precision0, recall0, precision1, recall1, f1_0, f1_1, macro_f1


def evaluate_thresholds_for_records(records, split: str, n0: int, n1: int, thresholds):
    top_k_used = len(records)
    counts0, counts1 = bit_counts_to_match_counts(records, split, n0, n1)
    rows = []

    for threshold in thresholds:
        min_matches = threshold_to_min_matches(float(threshold), top_k_used)
        tp0 = int(np.count_nonzero(counts0 >= min_matches))
        fp0 = int(np.count_nonzero(counts1 >= min_matches))
        precision0, recall0, precision1, recall1, f1_0, f1_1, macro_f1 = compute_binary_metrics(tp0, fp0, n0, n1)
        rows.append({
            'threshold': float(threshold),
            'min_matches': int(min_matches),
            'top_k_used': int(top_k_used),
            'predicted0_on_label0': int(tp0),
            'predicted0_on_label1': int(fp0),
            'precision0': float(precision0),
            'recall0': float(recall0),
            'precision1': float(precision1),
            'recall1': float(recall1),
            'f1_0': float(f1_0),
            'f1_1': float(f1_1),
            'macro_f1': float(macro_f1),
        })
    return rows


def build_threshold_metrics_df(output_dir: Path, logic_names, lengths, top_hkms_per_length: int, top_k: int, split_totals, thresholds):
    rows = []
    for logic_name in logic_names:
        for length in lengths:
            records = load_top_hkms(output_dir, logic_name, length, top_hkms_per_length, top_k)
            for split_name in ('train', 'test'):
                n0, n1 = split_totals[split_name]
                metric_rows = evaluate_thresholds_for_records(records, split_name, n0, n1, thresholds)
                for row in metric_rows:
                    rows.append({
                        'logic_name': logic_name,
                        'logic_title': LOGIC_TITLES.get(logic_name, logic_name),
                        'length': int(length),
                        'split': split_name,
                        **row,
                    })
    return pd.DataFrame(rows)


def plot_threshold_curves(metrics_df: pd.DataFrame, split_name: str, lengths, logic_names, plot_config: dict):
    fig, axes = plt.subplots(len(lengths), len(logic_names), figsize=plot_config['figsize'], dpi=plot_config['dpi'], sharex=True, sharey=True)

    if len(lengths) == 1 and len(logic_names) == 1:
        axes = np.array([[axes]])
    elif len(lengths) == 1:
        axes = np.array([axes])
    elif len(logic_names) == 1:
        axes = np.array([[ax] for ax in axes])

    for row_idx, length in enumerate(lengths):
        for col_idx, logic_name in enumerate(logic_names):
            ax = axes[row_idx, col_idx]
            sub = metrics_df[(metrics_df['split'] == split_name) & (metrics_df['length'] == length) & (metrics_df['logic_name'] == logic_name)].sort_values('threshold')

            ax.plot(sub['threshold'], sub['precision0'], color=plot_config['precision_color'], linestyle=plot_config['label0_linestyle'], label='precision0')
            ax.plot(sub['threshold'], sub['recall0'], color=plot_config['recall_color'], linestyle=plot_config['label0_linestyle'], label='recall0')
            ax.plot(sub['threshold'], sub['precision1'], color=plot_config['precision_color'], linestyle=plot_config['label1_linestyle'], label='precision1')
            ax.plot(sub['threshold'], sub['recall1'], color=plot_config['recall_color'], linestyle=plot_config['label1_linestyle'], label='recall1')
            ax.plot(sub['threshold'], sub['macro_f1'], color=plot_config['macro_f1_color'], linewidth=2.0, label='macro_f1')

            ax.set_title(f"len={length} | {LOGIC_TITLES.get(logic_name, logic_name)}")
            ax.set_ylim(*plot_config['ylim'])
            if plot_config.get('grid', False):
                ax.grid(True, alpha=plot_config.get('grid_alpha', 0.3))

            if row_idx == len(lengths) - 1:
                ax.set_xlabel(plot_config['xlabel'])
            if col_idx == 0:
                ax.set_ylabel(plot_config['ylabel'])

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=5, frameon=False)
    fig.suptitle(f'Top-{TOP_K_FOR_CLASSIFIER} HKM threshold classifier | {split_name}', y=0.98)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    return fig


In [4]:
split_totals = load_split_totals()
split_totals


{'train': (864, 3630), 'test': (200, 925)}

In [5]:
metrics_df = build_threshold_metrics_df(
    output_dir=OUTPUT_DIR,
    logic_names=LOGIC_NAMES,
    lengths=LENGTHS,
    top_hkms_per_length=TOP_HKMS_PER_LENGTH,
    top_k=TOP_K_FOR_CLASSIFIER,
    split_totals=split_totals,
    thresholds=THRESHOLDS,
)

metrics_csv_path = RESULT_DIR / f'top{TOP_K_FOR_CLASSIFIER}_threshold_metrics_all_splits.csv'
metrics_df.to_csv(metrics_csv_path, index=False)

best_macro_f1_df = (
    metrics_df[metrics_df['split'] == 'test']
    .sort_values(['length', 'macro_f1', 'precision0', 'recall0', 'threshold', 'logic_name'], ascending=[True, False, False, False, True, True])
    .groupby(['length'], as_index=False)
    .first()
)
best_macro_f1_csv_path = RESULT_DIR / f'top{TOP_K_FOR_CLASSIFIER}_threshold_best_macro_f1.csv'
best_macro_f1_df.to_csv(best_macro_f1_csv_path, index=False)

metrics_df.head(24)


,logic_name,logic_title,length,split,threshold,min_matches,top_k_used,predicted0_on_label0,predicted0_on_label1,precision0,recall0,precision1,recall1
0,precision0,precision0,4,train,0.0,0,10,864,3630,0.192256,1.000000,0.000000,0.000000
1,precision0,precision0,4,train,0.1,1,10,324,26,0.925714,0.375000,0.869691,0.992837
2,precision0,precision0,4,train,0.2,2,10,7,0,1.000000,0.008102,0.809004,1.000000
3,precision0,precision0,4,train,0.3,3,10,0,0,0.000000,0.000000,0.807744,1.000000
4,precision0,precision0,4,train,0.4,4,10,0,0,0.000000,0.000000,0.807744,1.000000
5,precision0,precision0,4,train,0.5,5,10,0,0,0.000000,0.000000,0.807744,1.000000
6,precision0,precision0,4,train,0.6,6,10,0,0,0.000000,0.000000,0.807744,1.000000
7,precision0,precision0,4,train,0.7,7,10,0,0,0.000000,0.000000,0.807744,1.000000
8,precision0,precision0,4,train,0.8,8,10,0,0,0.000000,0.000000,0.807744,1.000000
9,precision0,precision0,4,train,0.9,9,10,0,0,0.000000,0.000000,0.807744,1.000000


In [6]:
plot_df = metrics_df[metrics_df['split'] == PLOT_SPLIT].copy()
plot_df.sort_values(['length', 'logic_name', 'threshold'])[['length', 'logic_name', 'threshold', 'top_k_used', 'min_matches']].head(24)


,length,logic_name,threshold,top_k_used,min_matches
55,2,precision0,0.0,10,0
56,2,precision0,0.1,10,1
57,2,precision0,0.2,10,2
58,2,precision0,0.3,10,3
59,2,precision0,0.4,10,4
60,2,precision0,0.5,10,5
61,2,precision0,0.6,10,6
62,2,precision0,0.7,10,7
63,2,precision0,0.8,10,8
64,2,precision0,0.9,10,9


In [7]:
fig = plot_threshold_curves(
    metrics_df=metrics_df,
    split_name=PLOT_SPLIT,
    lengths=LENGTHS,
    logic_names=LOGIC_NAMES,
    plot_config=PLOT_CONFIG,
)

plot_path = RESULT_DIR / f'top{TOP_K_FOR_CLASSIFIER}_{PLOT_SPLIT}_threshold_curves.png'
fig.savefig(plot_path, bbox_inches='tight')
plot_path


WindowsPath('D:/PythonProject/paper/hkm_top_atoms_ranked_outputs/atoms_per_feature_30_top_hkms_5000/top10_threshold_classifier/top10_test_threshold_curves.png')

In [8]:
fig


<Figure size 2240x2240 with 6 Axes>